In [1]:
# ============================================================
# GPT-2 Fine-Tuning with LoRA for Creative Story Generation
# ============================================================

# SECTION 1: Install Required Libraries
!pip install -q transformers datasets peft accelerate torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00


In [2]:
import os
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)

print("✅ All libraries imported successfully.")
print(f"🖥️  Device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")

✅ All libraries imported successfully.
🖥️  Device: CPU


In [3]:
DATASET_PATH = "stories.txt"  # <-- Replace with your uploaded filename

# ── DEMO: Create a sample stories.txt if it doesn't exist ──────────────────
if not os.path.exists(DATASET_PATH):
    sample_stories = """Once upon a time, in a land where rivers ran silver and trees whispered ancient secrets,
a young girl discovered a door hidden beneath an old oak. She pushed it open and stepped into a world
where stars fell like rain and dreams walked on two legs. She never looked back.

A forgotten kingdom hidden inside a mountain had slept for a thousand years. Its halls were carved from
obsidian and lit by veins of glowing crystal. When the last heir finally returned, she pressed her palm
to the gate and the whole mountain shook with a long-buried sigh of recognition.

The sea gave back what it had taken. One morning, the fisherman found his lost boat beached on the shore,
filled not with water but with letters—hundreds of them, sealed with wax, addressed to people long dead.
He spent the rest of his life delivering them.

In the city of a thousand clocks, no two showed the same time. Citizens lived by their own hours,
meeting only by accident and memory. The clockmaker who had built them all had vanished, leaving behind
a single note: "Time was never meant to be agreed upon."

She planted a library in the desert. Every book she buried sprouted into a tree whose leaves, when read
in the right light, told a different story. Travelers came from far away just to sit in its shade and
remember things they had never lived.
"""
    with open(DATASET_PATH, "w") as f:
        f.write(sample_stories)
    print(f"📄 Sample dataset created at '{DATASET_PATH}'")
else:
    print(f"📄 Dataset found: '{DATASET_PATH}'")

📄 Sample dataset created at 'stories.txt'


In [4]:
# ────────────────────────────────────────────────────────────
# SECTION 4: Load and Convert Text File to HuggingFace Dataset
# ────────────────────────────────────────────────────────────

def load_stories_from_txt(filepath, min_length=30):
    """
    Reads the .txt file and splits it into individual story chunks.
    Stories are separated by blank lines. Short chunks are filtered out.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Split by double newlines (paragraph/story breaks)
    stories = [s.strip() for s in raw_text.split("\n\n") if len(s.strip()) >= min_length]
    print(f"📚 Loaded {len(stories)} story chunks from '{filepath}'")
    return stories

# Load stories and wrap into a HuggingFace Dataset dict format
story_texts = load_stories_from_txt(DATASET_PATH)
hf_dataset = Dataset.from_dict({"text": story_texts})

# Split into train/validation sets (90/10 split)
split_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"✅ Train samples : {len(train_dataset)}")
print(f"✅ Eval  samples : {len(eval_dataset)}")
print(f"🔍 Sample entry  : {train_dataset[0]['text'][:120]}...")

📚 Loaded 5 story chunks from 'stories.txt'
✅ Train samples : 4
✅ Eval  samples : 1
🔍 Sample entry  : The sea gave back what it had taken. One morning, the fisherman found his lost boat beached on the shore, 
filled not wi...


In [5]:
# ────────────────────────────────────────────────────────────
# SECTION 5: Load GPT-2 Tokenizer and Tokenize Dataset
# ────────────────────────────────────────────────────────────

MODEL_NAME  = "gpt2"          # Use "gpt2-medium" or "gpt2-large" for better results
MAX_LENGTH  = 256             # Max token length per example (increase if GPU VRAM allows)
OUTPUT_DIR  = "./gpt2-lora-stories"  # Directory to save model + adapters

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# GPT-2 has no padding token by default; set it to the EOS token
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    """
    Tokenizes each text sample with truncation and padding.
    Labels are set equal to input_ids so the model learns to predict next tokens.
    """
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    # For causal language modeling, labels = input_ids
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Apply tokenization across the dataset (batched for speed)
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval  = eval_dataset.map(tokenize_function,  batched=True, remove_columns=["text"])

# Set dataset format to PyTorch tensors
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

print(f"✅ Tokenization complete.")
print(f"🔍 Sample token IDs (first 10): {tokenized_train[0]['input_ids'][:10]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

✅ Tokenization complete.
🔍 Sample token IDs (first 10): tensor([ 464, 5417, 2921,  736,  644,  340,  550, 2077,   13, 1881])


In [6]:
# ────────────────────────────────────────────────────────────
# SECTION 6: Load GPT-2 Model and Apply LoRA Configuration
# ────────────────────────────────────────────────────────────

# Load the base GPT-2 model
base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# ── LoRA Configuration ──────────────────────────────────────
# LoRA injects small trainable rank-decomposition matrices into attention layers.
# This means we only train ~1-2% of the parameters, saving memory and time.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Causal language modeling (next-token prediction)
    r=8,                             # Rank of the update matrices (higher = more capacity)
    lora_alpha=32,                   # Scaling factor (alpha/r controls effective learning rate)
    lora_dropout=0.1,                # Dropout for regularization
    bias="none",                     # Don't train bias terms
    # Target the query and value projection layers in GPT-2's attention blocks
    target_modules=["c_attn"],       # GPT-2 uses "c_attn" for combined QKV projection
)

# Wrap the base model with LoRA adapters
model = get_peft_model(base_model, lora_config)

# Print a summary showing trainable vs frozen parameters
model.print_trainable_parameters()

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ LoRA model loaded and moved to: {device}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364
✅ LoRA model loaded and moved to: cpu


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [7]:
# ────────────────────────────────────────────────────────────
# SECTION 7: Data Collator
# ────────────────────────────────────────────────────────────

# DataCollatorForLanguageModeling handles dynamic padding and
# creates the appropriate input/label pairs for causal LM training.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # mlm=False → causal (autoregressive) language modeling, not masked
)

In [10]:
# ────────────────────────────────────────────────────────────
# SECTION 8: Training Configuration and Trainer Setup
# ────────────────────────────────────────────────────────────

# Detect if fp16 (half-precision) is supported (requires CUDA GPU)
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Core hyperparameters ──────────────────────────────
    num_train_epochs=3,               # Train for 3 full passes over the dataset
    learning_rate=2e-5,               # Fine-tuning learning rate

    # ── Batch sizes (small for Colab GPU memory constraints) ──
    per_device_train_batch_size=2,    # Samples per GPU per step
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch = 2 * 4 = 8

    # ── Precision and performance ─────────────────────────
    fp16=use_fp16,                    # Use half-precision on GPU to save memory & speed up

    # ── Evaluation and saving ─────────────────────────────
    eval_strategy="epoch",            # Updated from evaluation_strategy to eval_strategy
    save_strategy="epoch",            # Save checkpoint once per epoch
    load_best_model_at_end=True,      # Restore best checkpoint after training
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── Logging ───────────────────────────────────────────
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,                 # Log every 10 steps

    # ── Reproducibility ───────────────────────────────────
    seed=42,

    # ── Colab-specific optimizations ──────────────────────
    dataloader_pin_memory=False,      # Disable pinned memory (avoids Colab warnings)
    report_to="none",                 # Disable WandB / external logging
)

# Initialize the Trainer
# Note: Removing 'tokenizer=tokenizer' as it is redundant when provided in data_collator
# and was causing a TypeError in this environment.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("✅ Trainer configured. Ready to train.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer configured. Ready to train.


In [11]:
# ────────────────────────────────────────────────────────────
# SECTION 9: Train the Model
# ────────────────────────────────────────────────────────────

print("\n🚀 Starting LoRA fine-tuning...\n")
train_result = trainer.train()

# Log final training metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print("\n✅ Training complete!")
print(f"   Total steps     : {metrics.get('train_steps_per_second', 'N/A')}")
print(f"   Final train loss: {metrics.get('train_loss', 'N/A'):.4f}")


🚀 Starting LoRA fine-tuning...



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,4.363757
2,No log,4.362647
3,No log,4.361990


***** train metrics *****
  epoch                    =        3.0
  total_flos               =     1465GF
  train_loss               =     4.4978
  train_runtime            = 0:00:28.99
  train_samples_per_second =      0.414
  train_steps_per_second   =      0.103

✅ Training complete!
   Total steps     : 0.103
   Final train loss: 4.4978


In [12]:
# ────────────────────────────────────────────────────────────
# SECTION 10: Save the LoRA Adapters and Tokenizer
# ────────────────────────────────────────────────────────────

ADAPTER_SAVE_PATH = f"{OUTPUT_DIR}/lora-adapters"

# Save only the LoRA adapter weights (NOT the full model — much smaller file)
model.save_pretrained(ADAPTER_SAVE_PATH)

# Save the tokenizer alongside so inference is self-contained
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ LoRA adapters saved to: '{ADAPTER_SAVE_PATH}'")
print(f"💾 Adapter files:")
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    print(f"   {f:40s} {size / 1024:.1f} KB")


# ───────────────────────────────────────

✅ LoRA adapters saved to: './gpt2-lora-stories/lora-adapters'
💾 Adapter files:
   README.md                                5.0 KB
   adapter_config.json                      0.9 KB
   adapter_model.safetensors                1155.0 KB
   tokenizer_config.json                    0.3 KB
   tokenizer.json                           3474.6 KB


In [13]:
# ────────────────────────────────────────────────────────────
# SECTION 11: Load the Trained LoRA Model for Inference
# ────────────────────────────────────────────────────────────

print("\n🔄 Loading trained LoRA model for inference...")

# Step 1: Load the original base GPT-2 model (frozen weights)
inference_base = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# Step 2: Load and attach the LoRA adapter weights on top
inference_model = PeftModel.from_pretrained(inference_base, ADAPTER_SAVE_PATH)
inference_model.eval()  # Set to evaluation mode (disables dropout)
inference_model.to(device)

# Step 3: Load the tokenizer
inference_tokenizer = GPT2Tokenizer.from_pretrained(ADAPTER_SAVE_PATH)
inference_tokenizer.pad_token = inference_tokenizer.eos_token

print("✅ Model and adapters loaded for inference.")


🔄 Loading trained LoRA model for inference...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and adapters loaded for inference.


In [14]:
# ────────────────────────────────────────────────────────────
# SECTION 12: Inference — Generate Creative Stories
# ────────────────────────────────────────────────────────────

def generate_story(
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.9,
    top_k: int = 50,
    top_p: float = 0.95,
    num_return_sequences: int = 1,
    repetition_penalty: float = 1.2,
) -> list[str]:
    """
    Generates creative story continuations from a given prompt.
    """
    # Tokenize the prompt and move to device
    inputs = inference_tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
    ).to(device)

    with torch.no_grad():  # Disable gradient computation for faster inference
        output_ids = inference_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,              # Enable sampling (non-greedy)
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            num_return_sequences=num_return_sequences,
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    # Decode the generated token IDs back to text
    generated_stories = []
    for ids in output_ids:
        text = inference_tokenizer.decode(ids, skip_special_tokens=True)
        generated_stories.append(text)

    return generated_stories


# ── Run inference with the example prompt ───────────────────

PROMPT = "A forgotten kingdom hidden inside a mountain..."

print("\n" + "═" * 60)
print("📖 STORY GENERATION")
print("═" * 60)
print(f"🖊️  Prompt: {PROMPT}\n")

stories = generate_story(
    prompt=PROMPT,
    max_new_tokens=200,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    num_return_sequences=2,       # Generate 2 different story variants
    repetition_penalty=1.2,
)

for i, story in enumerate(stories, 1):
    print(f"── Story Variant {i} {'─' * 40}")
    print(story)
    print()

print("═" * 60)
print("✅ Story generation complete!")


# ────────────────────────────────────────────────────────────
# SECTION 13: (Optional) Merge LoRA Weights into Base Model
# ────────────────────────────────────────────────────────────

MERGED_SAVE_PATH = f"{OUTPUT_DIR}/merged-model"

print(f"\n🔀 Merging LoRA adapters into base model...")
merged_model = inference_model.merge_and_unload()  # Returns a standard GPT2LMHeadModel
merged_model.save_pretrained(MERGED_SAVE_PATH)
inference_tokenizer.save_pretrained(MERGED_SAVE_PATH)

print(f"✅ Merged model saved to: '{MERGED_SAVE_PATH}'")
print("   (This model can be loaded with GPT2LMHeadModel.from_pretrained() directly,")
print("    without needing the peft library at inference time.)\n")

print("🎉 All done! Your GPT-2 LoRA fine-tuned model is ready.")


════════════════════════════════════════════════════════════
📖 STORY GENERATION
════════════════════════════════════════════════════════════
🖊️  Prompt: A forgotten kingdom hidden inside a mountain...

── Story Variant 1 ────────────────────────────────────────
A forgotten kingdom hidden inside a mountain... The ghost of an ancient lich. A spectral orc knight wakes up! He is the first to flee his homeland in this nightmare as he has no hope and must make way for your aid with every possible means available
Brawl 2nd Edition (2017, Wizards) Author: Mark Rosewater This one's different than its predecessor because it brings new twists on old concepts from RPG rules like Blood Magic that we've seen before but are not yet familiar with or use by newer players through their games history pages; you can learn more here . It was my understanding then early 2016 would be when "The Dark Knight Returns" started releasing so I did add some updates at various points throughout 2015 about how they 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved to: './gpt2-lora-stories/merged-model'
   (This model can be loaded with GPT2LMHeadModel.from_pretrained() directly,
    without needing the peft library at inference time.)

🎉 All done! Your GPT-2 LoRA fine-tuned model is ready.
